<h1>🧬 ADSP — Step 2: the functional-consequence filter</h1>

Step 1 asked *is there a protein-coding gene this variant could affect?* This
one asks *and does it actually do something to it?*

**In one line:** each variant goes down exactly one of two branches, and has to
earn its place there.

| branch | for | criterion |
| --- | --- | --- |
| **A** | coding variants | must alter the protein |
| **B** | non-coding variants | must be a brain **eQTL and pQTL for the same gene** |

One row of the product is a **(variant, gene)** pair, where the gene is the one
the variant will be paired on in step 3.

## It is a partition, not two filters

> "Okay, so the new plan is to prioritize functional consequence, with differing
> definitions of 'functional' criteria for noncoding and coding variants."
> — Diane Xue, 2026-09-03
>
> "Yes, it should be a partition I think. There are different inclusion criteria
> for coding and non-coding variants." — Diane Xue, answering the question below

The distinction matters and it was asked explicitly. A coding variant with
strong regulatory evidence is **not** tested against the regulatory criterion —
it goes to branch A, and if it does not alter the protein it leaves the study.
Under two independent filters it would have been rescued by branch B.

## The gene a branch B variant pairs on

> "We want to keep all noncoding variants where the eQTL and pQTL point to the
> same gene, **regardless of what gene the variant actually sits in**."
> — Diane Xue

A regulatory variant sits *inside* one gene and acts *on* another, and they are
usually different — that is what makes it regulatory. The variant was selected
because it affects the target gene's expression and protein level, so the target
is what it pairs on. The gene it happens to sit in is not carried forward.

## Where the data comes from

| | source |
| --- | --- |
| consequences | the bundle, via step 1's product |
| eQTL | `expand_variant_regulatory` — GTEx, **13 brain tissues**, eQTL only |
| pQTL | the FunGen xQTL file, read directly — **not in BF4** |

That last row is the one to remember: this notebook is the only place that knows
where the pQTL data came from. If the requirement outlives this analysis it
should become a DTP, the way GTEx eQTL is.

### 1. Open a bundle, and read step 1

In [1]:
import json
import time
from pathlib import Path

import pandas as pd

from biofilter import Biofilter

bf = Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
ADSP = _root / "notebooks" / "Andre" / "adsp"
DATA_DIR, OUTPUT_DIR = ADSP / "data", ADSP / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STEP1_PATH = OUTPUT_DIR / "step1_per_variant_coding.csv"
PQTL_PATH = DATA_DIR / "brain_pQTL_hmt_variant_gene_lookup.tsv.gz"

step1 = pd.read_csv(STEP1_PATH)
print(f"{len(step1):,} rows over {step1.variant.nunique():,} variants from step 1")
step1.head(3)

[INFO] ════════════════════════════════════

[INFO] 🚀 Initializing Biofilter

[INFO]    • Version: 4.3.0

[INFO]    • Debug mode: False

[INFO]    • Config: /Users/andrerico/Works/Sys/biofilter_430/.biofilter.toml

...

[INFO]    • Views:  37 (read-only)

[INFO]    • Time:   161.4 ms

[INFO] ════════════════════════════════════

380,775 rows over 355,644 variants from step 1

,variant,chromosome,position,coding_gene,gene_entity_id,gene_ensembl_id,relation_to_gene,consequence,consequence_category,severity_rank,gene_locus_group,gene_locus_type,n_coding_genes
0,1:702358:G:A,1,702358,OR4F16,18620,ENSG00000284662,within gene,intron_variant,non_coding,28,protein-coding gene,gene with protein product,1
1,1:722408:G:C,1,722408,OR4F16,18620,ENSG00000284662,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1
2,1:919049:G:A,1,919049,SAMD11,31010,ENSG00000187634,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1


**Check that step 1 came from this bundle before joining anything.** Step 1's
`gene_entity_id` values only mean something next to the build that produced
them. A product built from another bundle would join against a different id
space and return rows that look perfectly fine.

This is what the `.provenance.json` sidecar is for, and it is worth two lines.

In [2]:
BUNDLE_ID = bf.report.run("platform_etl_status").provenance["bundle_id"]
step1_bundle = json.loads(Path(f"{STEP1_PATH}.provenance.json").read_text())["bundle_id"]

print(f"step 1 was built from : {step1_bundle}")
print(f"this bundle is        : {BUNDLE_ID}")
assert step1_bundle == BUNDLE_ID, "re-run step 1 against this bundle"
print("same bundle — safe to join on gene_entity_id")

[INFO] Report 'platform_etl_status' produced 68 rows in 0.02s from bundle 9b8419b48be5004e.

step 1 was built from : 9b8419b48be5004e

this bundle is        : 9b8419b48be5004e

same bundle — safe to join on gene_entity_id

### 2. The partition

A variant is **coding** if it has a coding consequence. The bundle already
carries that judgement as `consequence_category`, so there is no list of
consequence names to maintain here.

One thing worth checking rather than assuming: the product keeps only the *most
severe* consequence per (variant, gene), so "the category of the most severe
row" and "has a coding consequence anywhere" could in principle disagree. They
do not, and not by luck — VEP's severity ordering puts every coding consequence
above every non-coding one, so if a variant has a coding consequence at all, its
most severe consequence is coding.

In [3]:
coding_rows = step1[step1.consequence_category.eq("coding")]
coding_variants = set(coding_rows.variant)
non_coding = sorted(set(step1.variant) - coding_variants)

print(step1.consequence_category.value_counts().to_string())
print(f"\ncoding variants     -> branch A : {len(coding_variants):,}")
print(f"non-coding variants -> branch B : {len(non_coding):,}")

consequence_category
non_coding    323017
regulatory     51148
coding          6610


coding variants     -> branch A : 6,601

non-coding variants -> branch B : 349,043

### 3. Branch A — coding variants that alter the protein

Diane listed ten consequences. They map exactly onto **VEP severity rank ≤ 14**,
so the code uses the rank: one number, instead of a list of ten strings to keep
in sync with the bundle's vocabulary.

| rank | consequence | | rank | consequence |
|---:|---|---|---:|---|
| 2 | splice_acceptor_variant | | 7 | start_lost |
| 3 | splice_donor_variant | | 11 | inframe_insertion |
| 4 | stop_gained | | 12 | inframe_deletion |
| 5 | frameshift_variant | | 13 | missense_variant |
| 6 | stop_lost | | 14 | protein_altering_variant |

In [4]:
PROTEIN_ALTERING_MAX_RANK = 14

branch_a = (
    coding_rows[coding_rows.severity_rank.le(PROTEIN_ALTERING_MAX_RANK)]
    .drop_duplicates(["variant", "gene_entity_id"])
    .copy()
)
print(f"{branch_a.variant.nunique():,} variants, {len(branch_a):,} variant x gene rows\n")
print(coding_rows.groupby(["consequence", "severity_rank"]).variant.nunique()
      .sort_index(level=1).to_string())

2,800 variants, 2,802 variant x gene rows


consequence                          severity_rank
splice_acceptor_variant              2                  36
splice_donor_variant                 3                  38
stop_gained                          4                  45
stop_lost                            6                  17
start_lost                           7                  16
missense_variant                     13               2648
splice_donor_5th_base_variant        15                 35
splice_region_variant                16                476
splice_donor_region_variant          17                 96
splice_polypyrimidine_tract_variant  18                458
stop_retained_variant                21                  5
synonymous_variant                   22               2733

**Four of the ten consequences produce nothing.** `frameshift_variant`,
`inframe_insertion`, `inframe_deletion` and `protein_altering_variant` are indel
consequences, and the ADSP list is SNV-only. Worth confirming that is by
construction rather than indels having been lost upstream.

**The partition costs 3,801 coding variants**, and they are worth looking at
rather than filing away. 2,732 are synonymous — they do not change the protein,
which is the criterion working as intended. But **1,064 are splice-adjacent**:
`splice_region_variant` (475), `splice_polypyrimidine_tract_variant` (458),
`splice_donor_region_variant` (96) and `splice_donor_5th_base_variant` (35).

These sit just outside the two splice consequences on the list. They are coding
by category, so the partition sends them to branch A, where they fail. They are
not eligible for branch B either, whatever regulatory evidence they carry. If
that is not intended, the fix is one more rank in the branch A cutoff — but that
is a decision for Diane, not a default to set here.

In [5]:
dropped_a = coding_rows[~coding_rows.variant.isin(branch_a.variant)]
print(dropped_a.groupby("consequence").variant.nunique().sort_values(ascending=False).to_string())

consequence
synonymous_variant                     2732
splice_region_variant                   475
splice_polypyrimidine_tract_variant     458
splice_donor_region_variant              96
splice_donor_5th_base_variant            35
stop_retained_variant                     5

### 4. Branch B — brain eQTL **and** pQTL on the same gene

Two sources have to agree about the same variant and the same gene: the variant
changes how much RNA that gene makes (eQTL), **and** how much protein it makes
(pQTL).

`expand_variant_regulatory` answers the eQTL half from the bundle. Note what it
returns: `position_gene_symbol` is the gene the variant sits in,
`regulated_gene_id` is the gene it acts on. Branch B is about the second one.

In [6]:
started = time.perf_counter()
reg_result = bf.report.run("expand_variant_regulatory", input_data=non_coding)
reg = reg_result.to_pandas()
print(f"{len(reg):,} rows in {time.perf_counter() - started:.1f}s")

eqtl = reg[reg.status.eq("ok") & reg.regulated_gene_id.notna()]
print(f"eQTL links   : {len(eqtl):,}")
print(f"variants     : {eqtl.input_value.nunique():,}")
print(f"target genes : {eqtl.regulated_gene_id.nunique():,}")
print(f"tissues      : {eqtl.bio_context.nunique()} | qtl types: {sorted(eqtl.qtl_type.unique())}")

[INFO] Report 'expand_variant_regulatory' produced 860,432 rows in 180.32s from bundle 9b8419b48be5004e.

860,432 rows in 180.6s

eQTL links   : 593,608

variants     : 82,219

target genes : 17,324

tissues      : 13 | qtl types: ['eQTL']

**How often is the regulated gene a different gene?** This is the reason branch
B pairs on the target rather than on the gene the variant sits in.

⚠️ **The two gene columns are in different id spaces**, and comparing them
directly is a trap that silently returns "always different":
`position_gene_id` is a **BF4 entity id** (`11450` for APOE), while
`regulated_gene_id` is the **Ensembl id** GTEx names its target by
(`ENSG00000130204`). The comparison that means something is on the symbols.

This is the same mismatch step 3 has to resolve before the two branches can be
pooled — branch A hands it entity ids, branch B hands it Ensembl ids.

In [7]:
named = eqtl[eqtl.position_gene_symbol.notna() & eqtl.regulated_gene_symbol.notna()]
different = named.position_gene_symbol.ne(named.regulated_gene_symbol)

print(f"eQTL links                                   : {len(eqtl):,}")
print(f"  variant sits outside every gene body       : {int(eqtl.position_gene_symbol.isna().sum()):,}")
print(f"  target gene has no symbol in this bundle   : {int(eqtl.regulated_gene_symbol.isna().sum()):,}")
print(f"  both named                                 : {len(named):,}")
print(f"\nof those, regulating a DIFFERENT gene        : {int(different.sum()):,} "
      f"({100 * different.mean():.1f}%)")

eQTL links                                   : 593,608

  variant sits outside every gene body       : 80,127

  target gene has no symbol in this bundle   : 74,774

  both named                                 : 450,199


of those, regulating a DIFFERENT gene        : 334,433 (74.3%)

The pQTL half comes from a file. It carries a `chr` prefix on the variant id
that the bundle does not, and names its target by Ensembl id — which is what
`regulated_gene_id` is, so that is the join.

In [8]:
pqtl = pd.read_csv(PQTL_PATH, sep="\t")
pqtl["variant"] = pqtl.variant_id.str.replace(r"^chr", "", regex=True)

print(f"{len(pqtl):,} rows | {pqtl.variant.nunique():,} variants | "
      f"{pqtl.target_ensembl_id.nunique():,} target genes")
pqtl.head(3)

308,238 rows | 283,029 variants | 3,115 target genes

,variant_id,target_gene_symbol,target_ensembl_id,n_pqtl_cohorts,pqtl_cohorts,brain_regions,min_pval,min_FDR,min_hmt_p_adj_all,variant
0,chr1:39739455:TAAAG:T,PPIE,ENSG00000084072,2,Knight-ADRC;MSBB,PC;PHG,8.459907e-10,6.484909e-07,5.574232e-06,1:39739455:TAAAG:T
1,chr1:44898596:TA:T,AKR1A1,ENSG00000117448,2,Knight-ADRC;ROSMAP,DLPFC;PC,5.152940e-19,1.964075e-16,6.285041e-15,1:44898596:TA:T
2,chr1:45379897:AT:A,AKR1A1,ENSG00000117448,2,Knight-ADRC;ROSMAP,DLPFC;PC,4.431183e-18,1.193699e-15,3.403591e-14,1:45379897:AT:A


In [9]:
both = eqtl.merge(
    pqtl,
    left_on=["input_value", "regulated_gene_id"],
    right_on=["variant", "target_ensembl_id"],
    how="inner",
)

branch_b = (
    both.groupby(["input_value", "regulated_gene_id"], as_index=False)
    .agg(
        pair_gene_symbol=("regulated_gene_symbol", "first"),
        chromosome=("chromosome", "first"),
        position=("position", "first"),
        n_eqtl_tissues=("bio_context", "nunique"),
        eqtl_p_min=("p_value", "min"),
        n_pqtl_cohorts=("n_pqtl_cohorts", "max"),
        pqtl_fdr_min=("min_FDR", "min"),
    )
    .rename(columns={"input_value": "variant", "regulated_gene_id": "pair_gene_id"})
)


# The QTL target arrives as an Ensembl id. Resolving it to a BF4 entity id
# belongs here, where the gene is chosen, rather than in step 3: both branches
# then leave step 2 carrying the same two identifiers.
resolved_targets = bf.report.run(
    "annotate_gene",
    input_data=branch_b.pair_gene_id.dropna().unique().tolist(),
    include_variant_summary=False,
).to_pandas()
branch_b = branch_b.merge(
    resolved_targets.loc[resolved_targets.entity_id.notna(), ["input_value", "entity_id"]]
    .rename(columns={"input_value": "pair_gene_id", "entity_id": "pair_gene_entity_id"}),
    on="pair_gene_id", how="left",
)
print(f"target genes with no entity in this bundle: "
      f"{int(branch_b.pair_gene_entity_id.isna().sum())} rows")

print(f"{branch_b.variant.nunique():,} variants | "
      f"{branch_b.pair_gene_id.nunique():,} target genes | {len(branch_b):,} rows")
branch_b.head()

[INFO] Report 'annotate_gene' produced 1,023 rows in 0.14s from bundle 9b8419b48be5004e.

target genes with no entity in this bundle: 5 rows

4,803 variants | 1,023 target genes | 4,982 rows

,variant,pair_gene_id,pair_gene_symbol,chromosome,position,n_eqtl_tissues,eqtl_p_min,n_pqtl_cohorts,pqtl_fdr_min,pair_gene_entity_id
0,10:100046569:C:T,ENSG00000095485,CWF19L1,10.0,100046569.0,11,5.180561e-17,1,1.258148e-07,44256.0
1,10:100057584:A:G,ENSG00000095485,CWF19L1,10.0,100057584.0,12,1.648797e-24,1,2.210175e-05,44256.0
2,10:100164661:T:C,ENSG00000095485,CWF19L1,10.0,100164661.0,13,1.761308e-45,2,1.289411e-18,44256.0
3,10:100251790:C:T,ENSG00000095485,CWF19L1,10.0,100251790.0,11,1.139784e-09,1,5.766111e-07,44256.0
4,10:100290587:A:T,ENSG00000196072,BLOC1S2,10.0,100290587.0,13,2.543887e-46,2,4.630350e-07,22655.0


### 5. What actually limits branch B

Not the criterion — the **pQTL coverage**. Worth stating plainly, because the
instinct when a branch comes back small is to loosen the filter, and here that
would not be the lever.

In [10]:
adsp_input = set(pd.read_csv(DATA_DIR / "adsp_variants.csv", header=None,
                             names=["v"]).v.str.strip())
pqtl_variants = set(pqtl.variant)

print(f"variants in the pQTL file          : {len(pqtl_variants):,}")
print(f"  ... also in the ADSP input       : {len(pqtl_variants & adsp_input):,}")
print(f"  ... surviving step 1             : {len(pqtl_variants & set(step1.variant)):,}")
print(f"  ... with an eQTL on the same gene: {branch_b.variant.nunique():,}")

is_snv = pqtl.variant.str.match(r"^[^:]+:\d+:[ACGT]:[ACGT]$")
print(f"\npQTL file that is indel / other    : {100 * (~is_snv).mean():.0f}% "
      f"(the ADSP list is SNV-only)")

variants in the pQTL file          : 283,029

  ... also in the ADSP input       : 12,331

  ... surviving step 1             : 9,451

  ... with an eQTL on the same gene: 4,803


pQTL file that is indel / other    : 19% (the ADSP list is SNV-only)

### 6. Two open questions, measured

Neither is decided here. Both were raised and left open, and numbers are more
useful than opinions.

**Does the eQTL p-value cutoff earn its place?** Diane, after seeing the first
estimates: *"Now I'm wondering if we even need that initial p-value cutoff
threshold."* The answer below is that at a loose threshold it does almost
nothing — the pQTL requirement is already the filter — and at a strict one it
halves the branch.

**Would "eQTL in ≥5 brain tissues" work instead of the pQTL requirement?** That
was option "1B" in the 2026-09-04 meeting, and the hope was not to need it,
because choosing tissues is arbitrary. It is also much weaker: ≥5 tissues keeps
5x more variants than requiring a pQTL.

In [11]:
pair_p = both.groupby(["input_value", "regulated_gene_id"]).p_value.min().reset_index()
print("eQTL p-value cutoff, applied to branch B:")
for threshold in (None, 1e-4, 1e-6, 1e-8, 1e-10):
    sub = pair_p if threshold is None else pair_p[pair_p.p_value.le(threshold)]
    label = "no cutoff" if threshold is None else f"p <= {threshold:g}"
    print(f"  {label:<12} {sub.input_value.nunique():>6,} variants  "
          f"{sub.regulated_gene_id.nunique():>5,} genes")

tissues = (eqtl.groupby(["input_value", "regulated_gene_id"])
           .bio_context.nunique().reset_index(name="n"))
print("\nalternative '1B' — eQTL in N+ brain tissues, no pQTL requirement:")
for n in (1, 3, 5, 8, 13):
    sub = tissues[tissues.n.ge(n)]
    print(f"  >= {n:>2} tissues {sub.input_value.nunique():>7,} variants  "
          f"{sub.regulated_gene_id.nunique():>5,} genes")

eQTL p-value cutoff, applied to branch B:

  no cutoff     4,803 variants  1,023 genes

  p <= 0.0001   4,704 variants  1,008 genes

  p <= 1e-06    3,328 variants    799 genes

  p <= 1e-08    2,378 variants    642 genes

...

  >=  1 tissues  82,219 variants  17,324 genes

  >=  3 tissues  37,056 variants  8,293 genes

  >=  5 tissues  25,179 variants  5,822 genes

  >=  8 tissues  15,826 variants  3,860 genes

  >= 13 tissues   4,689 variants  1,325 genes

### 7. The product

Both branches, one table, with the branch on every row. `pair_gene_*` is the
gene step 3 will pair on — the gene itself for branch A, the QTL target for
branch B.

**Each gene leaves with two identifiers, and that is deliberate.**

| column | what it is |
| --- | --- |
| `pair_gene_entity_id` | this bundle's **join key**; meaningless outside the build that minted it |
| `pair_gene_id` | the **Ensembl id** — portable, and what a resolver understands |

Carrying both is the lesson of a bug worth remembering: an entity id is not an
identifier you hand to something that resolves names. 14,335 bare-number aliases
in this bundle are also the entity id of a *different* gene, so a resolver fed an
entity id can return the wrong gene and say nothing. Step 3 names which space it
is using, and this is where both become available.

The two branches carry different evidence columns, and that is deliberate too: a
protein-altering variant has a consequence and a severity, a regulatory one has
tissues and p-values. Null in one of those means **"not applicable to this
branch"**, not "not measured".

In [12]:
a_out = branch_a[["variant", "chromosome", "position", "coding_gene",
                  "gene_ensembl_id", "gene_entity_id",
                  "consequence", "severity_rank"]].rename(
    columns={"coding_gene": "pair_gene_symbol", "gene_ensembl_id": "pair_gene_id",
             "gene_entity_id": "pair_gene_entity_id"})
a_out.insert(3, "branch", "A_protein_altering")

b_out = branch_b[["variant", "chromosome", "position", "pair_gene_symbol",
                  "pair_gene_id", "pair_gene_entity_id", "n_eqtl_tissues",
                  "eqtl_p_min", "n_pqtl_cohorts", "pqtl_fdr_min"]].copy()
b_out.insert(3, "branch", "B_regulatory")

product = pd.concat([a_out, b_out], ignore_index=True).sort_values(
    ["chromosome", "position", "branch"]).reset_index(drop=True)

print(product.branch.value_counts().to_string())
print(f"\n{product.variant.nunique():,} variants in {len(product):,} rows")
print(f"rows carrying both identifiers: "
      f"{int((product.pair_gene_id.notna() & product.pair_gene_entity_id.notna()).sum()):,}")
product.head()

branch
B_regulatory          4982
A_protein_altering    2802


7,603 variants in 7,784 rows

rows carrying both identifiers: 7,779

,variant,chromosome,position,branch,pair_gene_symbol,pair_gene_id,pair_gene_entity_id,consequence,severity_rank,n_eqtl_tissues,eqtl_p_min,n_pqtl_cohorts,pqtl_fdr_min
0,1:966227:C:G,1.0,966227.0,A_protein_altering,PLEKHN1,ENSG00000187583,20895.0,missense_variant,13.0,NaN,NaN,NaN,NaN
1,1:973858:G:C,1.0,973858.0,A_protein_altering,PLEKHN1,ENSG00000187583,20895.0,missense_variant,13.0,NaN,NaN,NaN,NaN
2,1:973862:A:G,1.0,973862.0,A_protein_altering,PLEKHN1,ENSG00000187583,20895.0,missense_variant,13.0,NaN,NaN,NaN,NaN
3,1:973929:T:C,1.0,973929.0,A_protein_altering,PLEKHN1,ENSG00000187583,20895.0,missense_variant,13.0,NaN,NaN,NaN,NaN
4,1:974039:C:T,1.0,974039.0,A_protein_altering,PLEKHN1,ENSG00000187583,20895.0,missense_variant,13.0,NaN,NaN,NaN,NaN


### 8. Export

In [13]:
product_path = OUTPUT_DIR / "step2_selected_variants.csv"
product.to_csv(product_path, index=False)

Path(f"{product_path}.provenance.json").write_text(json.dumps({
    "step": "adsp_step_02_functional_filter",
    "bundle_id": BUNDLE_ID,
    "step1_input": str(STEP1_PATH),
    "pqtl_file": str(PQTL_PATH),
    "pqtl_in_bundle": False,
    "reports": ["expand_variant_regulatory"],
    "partition": {
        "A_protein_altering": "consequence_category=coding and severity_rank<=14",
        "B_regulatory": "consequence_category!=coding, brain eQTL and pQTL on the same gene",
    },
    "variants": int(product.variant.nunique()),
    "rows": len(product),
}, indent=2))

for path in sorted(OUTPUT_DIR.glob("step2_*")):
    print(f"{path.name:<45} {path.stat().st_size / 1e3:8.1f} KB")

step2_selected_variants.csv                      934.5 KB

step2_selected_variants.csv.provenance.json        0.6 KB

step2_summary.csv                                  0.3 KB

### 9. The same thing on the command line

```bash
python notebooks/Andre/adsp/step_02_adsp_functional_filter.py \
    --input   outputs/step1_per_variant_coding.csv \
    --pqtl    data/brain_pQTL_hmt_variant_gene_lookup.tsv.gz \
    --bundle  /project/hall_shared/datasets/biofilter/20260914 \
    --out-dir outputs
```

`--eqtl-p-max` applies the cutoff from §6. It is off by default.

### 10. Quick QA

In [14]:
overlap = set(a_out.variant) & set(b_out.variant)
checks = {
    "the branches do not overlap — it is a partition": not overlap,
    "every branch A variant is coding": bool(a_out.variant.isin(coding_variants).all()),
    "no branch B variant is coding": not b_out.variant.isin(coding_variants).any(),
    "every variant came from step 1": bool(product.variant.isin(set(step1.variant)).all()),
    "branch A rows are all protein-altering":
        bool(branch_a.severity_rank.le(PROTEIN_ALTERING_MAX_RANK).all()),
    "every branch B row names a target gene": not b_out.pair_gene_id.isna().any(),
    "both branches carry an Ensembl id": not product.pair_gene_id.isna().any(),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

print(f"\nbundle: {BUNDLE_ID}")

PASS  the branches do not overlap — it is a partition

PASS  every branch A variant is coding

PASS  no branch B variant is coding

PASS  every variant came from step 1

PASS  branch A rows are all protein-altering

PASS  every branch B row names a target gene

PASS  both branches carry an Ensembl id


bundle: 9b8419b48be5004e

### 11. Known gaps and what step 3 needs

**pQTL is read from a file, not from BF4.** This notebook is the only place that
records where it came from. If the requirement stays, it should be a DTP.

**The bundle carries 13 brain tissues and eQTL only.** No sQTL. Absence of
regulatory evidence here is absence *in those tissues*, not in biology.

**The ADSP list is autosomal**, reaching chromosomes 1-22. The bundle is the
complete build.

**381 splice-adjacent variants leave the study** — see §3. That is the partition
working as specified, and it may or may not be what was intended.

**Step 3 pairs these variants** through the genes in `pair_gene_symbol` /
`pair_gene_id`, using `pair_variants`. Note the two branches hand it different
id spaces: branch A gives a BF4 `gene_entity_id`, branch B gives an Ensembl id
from GTEx. Step 3 has to resolve the second one before the two can be pooled.